# KFold, GroupKFold hay chia theo thời gian?

Cùng một model, cùng một dữ liệu, ba cách cross-validation cho ra ba con số khác nhau. Con số nào
gần với điểm model sẽ đạt khi chạy thật (trên đơn **tương lai**, của khách **mới**)?

Bài này dùng bảng Olist của bài Intermediate 02 (bài toán review xấu, dự đoán lúc đơn vừa giao) và
hai bộ dữ liệu mô phỏng trong test.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` đọc dữ liệu thật, in kết quả. Test bỏ qua chúng.
- Chấm: `uv run pytest tracks/data-science/modules/10-feature-engineering/exercises/intermediate/03-time-group-cv -q`

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

In [ ]:
# Cell viết sẵn: đọc 8 bảng Olist, tạo một dòng cho mỗi đơn giao xong có review.
# Thời điểm dự đoán: lúc đơn vừa giao xong (order_delivered_customer_date), trước khi khách review.
DATES = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
         "order_delivered_customer_date", "order_estimated_delivery_date"]
ZIP = {"customer_zip_code_prefix": str, "seller_zip_code_prefix": str, "geolocation_zip_code_prefix": str}


def _haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = (np.radians(v) for v in (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))


def load_olist_table(raw) -> pd.DataFrame:
    raw = Path(raw)
    orders = pd.read_csv(raw / "olist_orders_dataset.csv", parse_dates=DATES)
    reviews = pd.read_csv(raw / "olist_order_reviews_dataset.csv",
                          parse_dates=["review_creation_date", "review_answer_timestamp"])
    customers = pd.read_csv(raw / "olist_customers_dataset.csv", dtype=ZIP)
    items = pd.read_csv(raw / "olist_order_items_dataset.csv")
    products = pd.read_csv(raw / "olist_products_dataset.csv")
    sellers = pd.read_csv(raw / "olist_sellers_dataset.csv", dtype=ZIP)
    payments = pd.read_csv(raw / "olist_order_payments_dataset.csv")
    geo = pd.read_csv(raw / "olist_geolocation_dataset.csv", dtype=ZIP)

    delivered = orders[(orders.order_status == "delivered") & orders.order_delivered_customer_date.notna()]
    latest = reviews.sort_values("review_answer_timestamp").drop_duplicates("order_id", keep="last")
    df = delivered.merge(latest, on="order_id").merge(customers, on="customer_id")
    df["bad_review"] = (df.review_score <= 2).astype(int)

    day = pd.Timedelta(days=1)
    df["purchase_hour"] = df.order_purchase_timestamp.dt.hour
    df["purchase_dow"] = df.order_purchase_timestamp.dt.day_of_week
    df["promised_days"] = (df.order_estimated_delivery_date - df.order_purchase_timestamp) / day
    df["delivery_days"] = (df.order_delivered_customer_date - df.order_purchase_timestamp) / day
    df["delay_days"] = (df.order_delivered_customer_date - df.order_estimated_delivery_date) / day
    df["approve_hours"] = (df.order_approved_at - df.order_purchase_timestamp) / pd.Timedelta(hours=1)

    items = items.merge(products, on="product_id", how="left").merge(sellers, on="seller_id", how="left")
    per_order = items.groupby("order_id").agg(
        n_items=("order_item_id", "size"), n_sellers=("seller_id", "nunique"),
        price=("price", "sum"), freight=("freight_value", "sum"), weight_g=("product_weight_g", "sum"),
        photos=("product_photos_qty", "mean"), desc_len=("product_description_lenght", "mean"))
    first = items[items.order_item_id == 1][
        ["order_id", "seller_id", "product_category_name", "seller_state", "seller_zip_code_prefix"]]
    df = df.merge(per_order, on="order_id").merge(first, on="order_id")
    df["freight_ratio"] = df.freight / (df.price + df.freight)
    main_payment = payments.sort_values("payment_value", ascending=False).drop_duplicates("order_id")
    df = df.merge(main_payment[["order_id", "payment_type", "payment_installments"]], on="order_id", how="left")

    ok = geo.geolocation_lat.between(-34, 6) & geo.geolocation_lng.between(-74, -34)
    centroid = geo[ok].groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].median()
    c = centroid.reindex(df.customer_zip_code_prefix).to_numpy()
    s = centroid.reindex(df.seller_zip_code_prefix).to_numpy()
    df["distance_km"] = _haversine_km(c[:, 0], c[:, 1], s[:, 0], s[:, 1])
    df["same_state"] = (df.customer_state == df.seller_state).astype(int)
    return df.sort_values("order_purchase_timestamp", kind="stable").reset_index(drop=True)


def time_split(df: pd.DataFrame) -> dict:
    t = df.order_purchase_timestamp
    return {"train": df[t < "2018-01-01"],
            "val": df[(t >= "2018-01-01") & (t < "2018-04-01")],
            "test": df[(t >= "2018-04-01") & (t < "2018-09-01")]}

In [ ]:
# Cell viết sẵn: model của bài Intermediate 02 (feature đã chọn, không leakage)
NUMERIC = [
    "purchase_hour", "purchase_dow", "promised_days", "delivery_days", "delay_days", "approve_hours",
    "n_items", "n_sellers", "price", "freight", "freight_ratio", "weight_g", "photos", "desc_len",
    "payment_installments", "distance_km", "same_state",
]
CATEGORICAL = ["customer_state", "seller_state", "payment_type", "product_category_name"]


def make_olist_model(high_card=()):
    parts = [
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="missing"),
                              OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20)), CATEGORICAL),
    ]
    if high_card:
        parts.append(("te", TargetEncoder(cv=KFold(5, shuffle=True, random_state=0)), list(high_card)))
    return make_pipeline(ColumnTransformer(parts), LogisticRegression(max_iter=2000))

In [ ]:
# explore: tìm gốc repo và đọc bảng feature Olist (tải bằng: uv run python data/download_olist.py)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
table = load_olist_table(ROOT / "data" / "raw")
splits = time_split(table)
train, val, test = splits["train"], splits["val"], splits["test"]
print(table.shape, {k: len(v) for k, v in splits.items()})
dev = pd.concat([train, val]).reset_index(drop=True)   # dữ liệu được phép nhìn khi phát triển model

## 1. Chia theo thời gian

`time_cv_splits(dates, n_splits=5, gap=0)` trả list `(train_idx, test_idx)` là **vị trí** (0..n−1)
trong `dates` như được truyền vào; `dates` có thể chưa sắp xếp. Sắp các dòng theo thời gian (sort
ổn định), rồi chia như `TimeSeriesSplit(n_splits, gap=gap)`: fold sau luôn ở tương lai so với
mọi dòng train của nó, và bỏ `gap` dòng ngay trước mỗi fold test.

In [ ]:
def time_cv_splits(dates, n_splits: int = 5, gap: int = 0) -> list[tuple[np.ndarray, np.ndarray]]:
    # TODO: thứ tự thời gian bằng np.argsort(..., kind="stable"), rồi TimeSeriesSplit trên thứ tự đó
    raise NotImplementedError

In [ ]:
# explore: mỗi fold test của dữ liệu dev nằm ở giai đoạn nào, tỉ lệ review xấu ra sao
for k, (tr, te) in enumerate(time_cv_splits(dev.order_purchase_timestamp)):
    d = dev.order_purchase_timestamp.iloc[te]
    print(k, len(tr), len(te), d.min().date(), "→", d.max().date(), "review xấu:", round(dev.bad_review.iloc[te].mean(), 3))

## 2. AUC theo từng fold và so ba cách chia

- `cv_auc(make_model, X, y, splits)`: với mỗi `(train_idx, test_idx)`, gọi `make_model()` lấy model
  **mới**, fit trên dòng train, tính ROC-AUC trên dòng test. Trả `np.ndarray` AUC theo thứ tự fold.
  `X` là DataFrame (dùng `.iloc`), `y` là mảng hoặc Series.
- `compare_schemes(make_model, X, y, dates, groups, n_splits=5, seed=0)`: DataFrame index
  `["kfold", "group_kfold", "time"]`, cột `mean`, `std` (std của AUC các fold, `np.std` mặc định).
  `kfold` là `KFold(n_splits, shuffle=True, random_state=seed)`, `group_kfold` là
  `GroupKFold(n_splits)` theo `groups`, `time` là `time_cv_splits(dates, n_splits)`.

In [ ]:
def cv_auc(make_model, X: pd.DataFrame, y, splits) -> np.ndarray:
    # TODO
    raise NotImplementedError


def compare_schemes(make_model, X: pd.DataFrame, y, dates, groups, n_splits: int = 5, seed: int = 0) -> pd.DataFrame:
    # TODO: ba danh sách split, mỗi danh sách qua cv_auc
    raise NotImplementedError

In [ ]:
# explore: ba cách chia trên Olist (dev = train + val), rồi điểm thật trên test
y_dev = dev.bad_review.to_numpy()
print(compare_schemes(make_olist_model, dev, y_dev, dev.order_purchase_timestamp, dev.customer_unique_id).round(4))
print("AUC từng fold theo thời gian:",
      cv_auc(make_olist_model, dev, y_dev, time_cv_splits(dev.order_purchase_timestamp)).round(3))
final = make_olist_model().fit(dev, y_dev)
print("AUC test (2018-04 → 2018-08):", round(roc_auc_score(test.bad_review, final.predict_proba(test)[:, 1]), 4))

## 3. Feature cấp khách và GroupKFold

Thêm target encoding của `customer_unique_id` (một feature "nhớ" khách). Với KFold, đơn khác của
cùng khách nằm ở fold train; với GroupKFold thì không. Khách Olist hiếm khi mua lại, nên chênh lệch
nhỏ; trên dữ liệu mô phỏng trong test (mỗi khách 10 dòng) chênh lệch rất lớn.

In [ ]:
# explore: số đơn thuộc khách có từ 2 đơn trở lên, và KFold vs GroupKFold khi model "nhớ" khách
per_customer = dev.groupby("customer_unique_id").size()
print("tỉ lệ đơn của khách mua lại:", round(per_customer[per_customer > 1].sum() / len(dev), 4))
memorise = lambda: make_olist_model(high_card=["customer_unique_id"])
print(compare_schemes(memorise, dev, y_dev, dev.order_purchase_timestamp, dev.customer_unique_id).round(4))

## 4. Viết lại cho đồng nghiệp (không chấm tự động)

Trả lời ngắn trong cell này: (1) báo cáo AUC nào cho model này, kèm khoảng nào; (2) vì sao `time`
có độ lệch chuẩn lớn hơn hẳn `kfold`; (3) nếu sản phẩm phục vụ chủ yếu khách mới thì cách chia nào
đúng; (4) AUC test thấp hơn mọi con số CV: lỗi của CV hay của thế giới?